# 06 — Agent: escalation + grounded generation

Compose the frozen classifier (`04`) and frozen retriever (`05`) into an
end-to-end agent that either **escalates** or **generates a grounded reply**,
then judge the result.

## Three independent evaluation questions

- **Q1 — Grounding.** Does retrieved context improve reply quality?
- **Q2 — Generation.** Are replies correct, relevant, helpful, and supported?
- **Q3 — Automation.** Can the system safely automate useful traffic?

## Hard rules

1. All upstream SHAs verified in Cell 2 (taxonomy, golden, classifier,
   retriever, corpus, rubrics).
2. Classifier and retriever test IDs must equal golden test IDs.
3. **No re-running classifier or retrieval.** 06 reads frozen JSONL only.
4. **Escalation-suitability judge sees customer text only.** No intent,
   confidence, retrieved context, or classifier output.
5. **Universal judge sees query + reply only.** Groundedness judge sees
   query + precedents + reply. Groundedness judge applies only to grounded
   replies.
6. Both judges are blinded to approach identity.
7. **Selection rule, predeclared:**
   - Safety gate: `unsafe_automation_rate ≤ 0.10`
   - Primary: mean universal score of non-escalated replies
   - Tie-break (Δ ≤ 0.30): higher automation coverage
   - Tie-break: simpler pipeline (`ungrounded < grounded_no_escalate < full_pipeline`)
8. `chosen_agent.json` immutable before test.
9. Test evaluated once. `TEST_LOCK.json` refuses re-run.
10. Escalation thresholds are tuned on dev only.
11. Judge rubrics are frozen before any judging. Three separate rubric SHAs.
12. Top-5 retrieval context, pinned by 05's dev analysis.

In [1]:
# ============ CELL 1b: Bootstrap ============
from pathlib import Path
import os, json, hashlib, sys, time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

def _find_root():
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").exists() or (parent / ".git").exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = _find_root()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BRAND           = "GWRHelp"
CONFIG_DIR      = PROJECT_ROOT / "configs"
TAX_DIR         = PROJECT_ROOT / "runs" / "taxonomy"
GS_DIR          = PROJECT_ROOT / "runs" / "golden_set"
RUNS_CLASSIFIER = PROJECT_ROOT / "runs" / "classifier"
RUNS_RETRIEVAL  = PROJECT_ROOT / "runs" / "retrieval"
RUNS_AGENT      = PROJECT_ROOT / "runs" / "agent"
CACHE_DIR       = PROJECT_ROOT / "cache"

for d in (RUNS_AGENT, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ImportError:
    pass

_has_key = bool(os.environ.get("GROQ_API_KEY") or os.environ.get("GROQ_API_KEYS"))
print("Bootstrap complete.")
print(f"  PROJECT_ROOT:     {PROJECT_ROOT}")
print(f"  RUNS_AGENT:       {RUNS_AGENT.relative_to(PROJECT_ROOT)}")
print(f"  GROQ key present: {_has_key}")
assert _has_key, "Set GROQ_API_KEY in .env"

Bootstrap complete.
  PROJECT_ROOT:     D:\CODIN PLAYGROUND\ML-AI\ResolveIQ
  RUNS_AGENT:       runs\agent
  GROQ key present: True


In [2]:
# ============ CELL 2: Verify frozen inputs ============
import yaml

INTENTS_YAML   = CONFIG_DIR / "intents.yaml"
INTENTS_SHA_F  = TAX_DIR / "intents.yaml.sha256"
GOLDEN_JSONL   = GS_DIR / "golden_set.jsonl"
GOLDEN_SHA_F   = GS_DIR / "golden_set.sha256"
GOLDEN_META    = GS_DIR / "golden_set.meta.json"

CLS_CHOSEN      = RUNS_CLASSIFIER / "chosen.json"
CLS_TEST_PREDS  = RUNS_CLASSIFIER / "predictions_test.jsonl"
CLS_RELIABILITY = RUNS_CLASSIFIER / "reliability_for_escalation.csv"

RET_CHOSEN      = RUNS_RETRIEVAL / "chosen_retriever.json"
RET_CORPUS      = RUNS_RETRIEVAL / "corpus.jsonl"
RET_CORPUS_SHA  = RUNS_RETRIEVAL / "corpus.sha256"
RET_TEST_PREDS  = RUNS_RETRIEVAL / "predictions_test.jsonl"

for p in (INTENTS_YAML, INTENTS_SHA_F, GOLDEN_JSONL, GOLDEN_SHA_F, GOLDEN_META,
          CLS_CHOSEN, CLS_TEST_PREDS, CLS_RELIABILITY,
          RET_CHOSEN, RET_CORPUS, RET_CORPUS_SHA, RET_TEST_PREDS):
    assert p.exists(), f"Missing required input: {p}"

# --- Taxonomy ---
tax_sha = hashlib.sha256(INTENTS_YAML.read_bytes()).hexdigest()
assert tax_sha == INTENTS_SHA_F.read_text(encoding="utf-8").strip(), \
    "Taxonomy SHA mismatch"
tax = yaml.safe_load(INTENTS_YAML.read_bytes())
assert tax.get("draft") is False
INTENT_NAMES = [i["name"] for i in tax["intents"]]

# --- Golden ---
golden_sha = hashlib.sha256(GOLDEN_JSONL.read_bytes()).hexdigest()
golden_meta = json.loads(GOLDEN_META.read_text(encoding="utf-8"))
assert golden_sha == golden_meta["golden_sha256"]
assert golden_sha == GOLDEN_SHA_F.read_text(encoding="utf-8").strip()
assert golden_meta["taxonomy_sha256"] == tax_sha

GOLDEN_TEST_IDS  = set(int(x) for x in golden_meta["test_root_ids"])
GOLDEN_DEV_SIZE  = int(golden_meta["dev_size"])
GOLDEN_TEST_SIZE = int(golden_meta["test_size"])

# --- Classifier ---
cls_chosen = json.loads(CLS_CHOSEN.read_text(encoding="utf-8"))
cls_chosen_sha = hashlib.sha256(CLS_CHOSEN.read_bytes()).hexdigest()
assert cls_chosen["taxonomy_sha256"] == tax_sha
assert cls_chosen["golden_sha256"]   == golden_sha
CLS_APPROACH = cls_chosen["approach"]

_cls_test_ids = set()
with open(CLS_TEST_PREDS, "r", encoding="utf-8") as f:
    for line in f:
        _cls_test_ids.add(int(json.loads(line)["root_id"]))
assert _cls_test_ids == GOLDEN_TEST_IDS, "Classifier test IDs != golden test IDs"

# --- Retriever ---
ret_chosen = json.loads(RET_CHOSEN.read_text(encoding="utf-8"))
ret_chosen_sha = hashlib.sha256(RET_CHOSEN.read_bytes()).hexdigest()
assert ret_chosen["taxonomy_sha256"]   == tax_sha
assert ret_chosen["golden_sha256"]     == golden_sha
assert ret_chosen["classifier_sha256"] == cls_chosen_sha
RET_APPROACH = ret_chosen["approach"]

CORPUS_SHA = hashlib.sha256(RET_CORPUS.read_bytes()).hexdigest()
assert CORPUS_SHA == RET_CORPUS_SHA.read_text(encoding="utf-8").strip()
assert ret_chosen["corpus_sha256"] == CORPUS_SHA

_ret_test_ids = set()
with open(RET_TEST_PREDS, "r", encoding="utf-8") as f:
    for line in f:
        _ret_test_ids.add(int(json.loads(line)["root_id"]))
assert _ret_test_ids == GOLDEN_TEST_IDS, "Retriever test IDs != golden test IDs"

# --- Reliability table (frozen from 04) ---
REL_TABLE = pd.read_csv(CLS_RELIABILITY, encoding="utf-8")

print(f"Taxonomy SHA:        {tax_sha[:16]}...")
print(f"Golden SHA:          {golden_sha[:16]}...")
print(f"Classifier:          {CLS_APPROACH}  (SHA {cls_chosen_sha[:12]}...)")
print(f"Retriever:           {RET_APPROACH}  (SHA {ret_chosen_sha[:12]}...)")
print(f"Corpus SHA:          {CORPUS_SHA[:16]}...")
print(f"Golden dev/test:     {GOLDEN_DEV_SIZE} / {GOLDEN_TEST_SIZE}")
print(f"Reliability table:   {len(REL_TABLE)} rows")

Taxonomy SHA:        7a05e4af68a50981...
Golden SHA:          acde341dc09e85ae...
Classifier:          llm_zeroshot  (SHA 7d51251a172e...)
Retriever:           tfidf  (SHA 33dbaea4a95a...)
Corpus SHA:          69327018136200df...
Golden dev/test:     59 / 139
Reliability table:   10 rows


In [3]:
# ============ CELL 3: Load frozen dev predictions ============
from support_agent.escalation.policy import calibrate_confidence

# Golden dev rows
dev_rows = []
with open(GOLDEN_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r["split"] == "dev":
            dev_rows.append(r)
dev_golden = pd.DataFrame(dev_rows)[["root_id", "customer_text",
                                     "true_intent", "_source"]].copy()
dev_golden["root_id"] = dev_golden["root_id"].astype(int)

# Classifier dev predictions
CLS_DEV_PREDS = RUNS_CLASSIFIER / CLS_APPROACH / "predictions_dev.jsonl"
assert CLS_DEV_PREDS.exists(), f"Missing: {CLS_DEV_PREDS}"
cls_dev = pd.read_json(CLS_DEV_PREDS, orient="records", lines=True)
cls_dev = cls_dev[["root_id", "pred_intent", "confidence",
                   "alternative_intent", "alternative_confidence"]].copy()
cls_dev["root_id"] = cls_dev["root_id"].astype(int)

# Retriever dev predictions
RET_DEV_PREDS = RUNS_RETRIEVAL / RET_APPROACH / "predictions_dev.jsonl"
assert RET_DEV_PREDS.exists(), f"Missing: {RET_DEV_PREDS}"
ret_dev = pd.read_json(RET_DEV_PREDS, orient="records", lines=True)
ret_dev = ret_dev[["root_id", "corpus_coverage", "top_ids", "top_scores"]].copy()
ret_dev["root_id"] = ret_dev["root_id"].astype(int)

# Join
dev = (dev_golden
       .merge(cls_dev, on="root_id", how="inner")
       .merge(ret_dev, on="root_id", how="inner"))

assert len(dev) == GOLDEN_DEV_SIZE, \
    f"Expected {GOLDEN_DEV_SIZE} dev rows, got {len(dev)}"

# Calibrated confidence via lookup
dev["confidence_calibrated"] = dev["confidence"].apply(
    lambda c: calibrate_confidence(c, REL_TABLE)
)

print(f"Dev rows: {len(dev)}")
print(dev[["root_id", "true_intent", "pred_intent", "confidence",
           "confidence_calibrated", "corpus_coverage"]].head(5).to_string(index=False))

Dev rows: 59
 root_id        true_intent        pred_intent  confidence  confidence_calibrated  corpus_coverage
 1267080   seat_reservation  praise_or_chatter        0.90               0.601504                1
 2022953 delay_compensation service_disruption        0.90               0.601504                1
 2703317 delay_compensation service_disruption        0.90               0.601504                1
 1897157      booking_issue      booking_issue        0.95               0.601504                1
 1320194   seat_reservation   seat_reservation        0.95               0.601504                1


In [4]:
# ============ CELL 4: Escalation policy ============
from support_agent.escalation.policy import EscalationConfig, decide

ESCALATION = EscalationConfig(
    tau_c=0.40,
    tau_m=0.15,
    escalate_on_other_or_ambiguous=True,
    escalate_on_zero_coverage=True,
)

# Preview on dev
preview = dev.apply(
    lambda r: pd.Series(decide(r, ESCALATION),
                        index=["escalated", "reason"]),
    axis=1,
)
prev_df = pd.concat([dev[["root_id"]], preview], axis=1)

print("Escalation preview on dev:")
print(prev_df["reason"].replace("", "generate").value_counts().to_string())
print(f"\nEscalation rate: {prev_df['escalated'].mean():.3f}")
print(f"Generate rate:   {(~prev_df['escalated']).mean():.3f}")

Escalation preview on dev:
reason
generate                     51
intent_other_or_ambiguous     7
corpus_coverage_zero          1

Escalation rate: 0.136
Generate rate:   0.864


In [6]:
# ============ CELL 4b: Threshold sweep on dev (documentation) ============
# Uses the escalation-suitability labels produced by Cell 11.
# Loads them from disk if not already in memory. Safe to run before or
# after Cell 11.

import itertools
import pandas as pd
from pathlib import Path
from support_agent.escalation.policy import EscalationConfig, decide

# --- Ensure esc_df is available ---
if "esc_df" not in dir():
    _esc_path = RUNS_AGENT / "escalation_suitability_dev.jsonl"
    assert _esc_path.exists(), (
        f"Missing {_esc_path}. Run Cell 11 first to produce "
        f"escalation-suitability labels on dev."
    )
    esc_df = pd.read_json(_esc_path, orient="records", lines=True)
    print(f"Loaded esc_df from disk: {len(esc_df)} rows")

if "dev" not in dir():
    raise RuntimeError("`dev` not defined. Run Cells 1b–3 first.")

TAU_C_GRID = [0.30, 0.35, 0.40, 0.45, 0.50]
TAU_M_GRID = [0.05, 0.10, 0.15, 0.20, 0.25]

esc_safe_dev = esc_df.set_index("root_id")["auto_handle_safe"]

sweep_rows = []
for tau_c, tau_m in itertools.product(TAU_C_GRID, TAU_M_GRID):
    cfg = EscalationConfig(
        tau_c=tau_c,
        tau_m=tau_m,
        escalate_on_other_or_ambiguous=True,
        escalate_on_zero_coverage=True,
    )
    decisions = dev.apply(
        lambda r: pd.Series(decide(r, cfg), index=["escalated", "reason"]),
        axis=1,
    )
    dev_w = dev.copy()
    dev_w["escalated"] = decisions["escalated"].values
    dev_w["auto_safe"] = dev_w["root_id"].map(esc_safe_dev)

    auto = dev_w[dev_w["escalated"] == False]
    n_auto = len(auto)
    unsafe = auto[auto["auto_safe"] == False]
    unsafe_rate = len(unsafe) / n_auto if n_auto else 0.0
    coverage = n_auto / len(dev_w)

    sweep_rows.append({
        "tau_c": tau_c,
        "tau_m": tau_m,
        "automation_coverage": coverage,
        "auto_n": n_auto,
        "unsafe_automation_rate": unsafe_rate,
        "unsafe_n": len(unsafe),
        "passes_gate": unsafe_rate <= 0.10,
    })

sweep = pd.DataFrame(sweep_rows)
sweep.to_csv(RUNS_AGENT / "threshold_sweep_dev.csv",
             index=False, quoting=1, encoding="utf-8")

passing = sweep[sweep["passes_gate"]].sort_values(
    "automation_coverage", ascending=False
)

print("Threshold sweep (dev):")
print(sweep.round(3).to_string(index=False))
print()
if len(passing):
    best = passing.iloc[0]
    print(f"Best passing operating point: "
          f"τ_c={best['tau_c']:.2f}  τ_m={best['tau_m']:.2f}  "
          f"coverage={best['automation_coverage']:.3f}  "
          f"unsafe={best['unsafe_automation_rate']:.3f}")
    print()
    print("NOTE: This operating point was NOT used in the current test run.")
    print("The current chosen agent is 'ungrounded' per the predeclared")
    print("fallback. This sweep documents what a tuned full_pipeline would")
    print("have selected had the sweep preceded Cell 13.")
else:
    print("No threshold combination in the grid passes the safety gate "
          "on dev.")

Loaded esc_df from disk: 59 rows
Threshold sweep (dev):
 tau_c  tau_m  automation_coverage  auto_n  unsafe_automation_rate  unsafe_n  passes_gate
  0.30   0.05                0.864      51                   0.706        36        False
  0.30   0.10                0.864      51                   0.706        36        False
  0.30   0.15                0.864      51                   0.706        36        False
  0.30   0.20                0.864      51                   0.706        36        False
  0.30   0.25                0.864      51                   0.706        36        False
  0.35   0.05                0.864      51                   0.706        36        False
  0.35   0.10                0.864      51                   0.706        36        False
  0.35   0.15                0.864      51                   0.706        36        False
  0.35   0.20                0.864      51                   0.706        36        False
  0.35   0.25                0.864      51  

In [7]:
# ============ CELL 5: Generation prompts ============
from support_agent.generation.generator import SYSTEM_PROMPT, build_user_message

print("=" * 72)
print("SYSTEM PROMPT (shared by ungrounded and grounded)")
print("=" * 72)
print(SYSTEM_PROMPT)
print()
print("=" * 72)
print("USER MESSAGE — UNGROUNDED")
print("=" * 72)
print(build_user_message("<customer message>", precedents=None))
print()
print("=" * 72)
print("USER MESSAGE — GROUNDED (example with 2 precedents)")
print("=" * 72)
_fake = [
    {"intent": "refund_request",
     "customer_text": "<example customer 1>",
     "response_text": "<example brand reply 1>"},
    {"intent": "delay_compensation",
     "customer_text": "<example customer 2>",
     "response_text": "<example brand reply 2>"},
]
print(build_user_message("<customer message>", precedents=_fake))

SYSTEM PROMPT (shared by ungrounded and grounded)
You are a customer support assistant for GWRHelp (Great Western Railway).

Draft a concise, helpful reply to the customer's message.

Rules:

- Be polite, brief, and specific to the customer's ask.
- Do NOT invent policies, compensation amounts, eligibility rules,
  timelines, procedures, contact routes, or operational actions.
- If historical precedents are provided below the customer message,
  use them as grounding evidence for any operational claims.
  Prefer phrasing supported by the precedents over generic phrasing.
- If you cannot answer from the information available, acknowledge
  the ask and direct the customer to the appropriate channel.
- Cite which precedent ranks you used (1-indexed) in the
  `used_precedent_ranks` field. Do not put citation markers in the
  reply text itself.
- Return strict JSON matching the required schema.

JSON schema:

{
  "reply": string,
  "used_precedent_ranks": array of integers,
  "grounded": bo

In [8]:
# ============ CELL 6: Judges ============
from support_agent.evaluation.judge import (
    Judge, ESCALATION_SYSTEM, UNIVERSAL_SYSTEM, GROUNDEDNESS_SYSTEM,
    ESCALATION_SCHEMA, UNIVERSAL_SCHEMA, GROUNDEDNESS_SCHEMA,
)

ESCALATION_RUBRIC_SHA   = hashlib.sha256(ESCALATION_SYSTEM.encode()).hexdigest()
UNIVERSAL_RUBRIC_SHA    = hashlib.sha256(UNIVERSAL_SYSTEM.encode()).hexdigest()
GROUNDEDNESS_RUBRIC_SHA = hashlib.sha256(GROUNDEDNESS_SYSTEM.encode()).hexdigest()

print("Rubric SHAs:")
print(f"  escalation:   {ESCALATION_RUBRIC_SHA[:16]}...")
print(f"  universal:    {UNIVERSAL_RUBRIC_SHA[:16]}...")
print(f"  groundedness: {GROUNDEDNESS_RUBRIC_SHA[:16]}...")
print()
print("=" * 72)
print("ESCALATION SUITABILITY RUBRIC")
print("=" * 72)
print(ESCALATION_SYSTEM)
print()
print("=" * 72)
print("UNIVERSAL RUBRIC")
print("=" * 72)
print(UNIVERSAL_SYSTEM)

Rubric SHAs:
  escalation:   620cc267a3666a8c...
  universal:    b649ea025980af73...
  groundedness: e43dce962cbbb76a...

ESCALATION SUITABILITY RUBRIC
You assess whether a customer message can be safely auto-handled.

You receive ONLY the customer's message. You do NOT receive any model
prediction, retrieved evidence, or taxonomy information.

A message is SAFE TO AUTO-HANDLE only if all of the following hold:
- The request is clear enough to answer without follow-up questions.
- The message contains the information needed, OR the answer does not
  require customer-specific details.
- A reasonable reply can be produced from general customer-service
  knowledge WITHOUT inventing company-specific policy, timelines,
  amounts, or actions.
- There is no legal, financial, medical, or vulnerability risk.
- It is a single, simple issue (not compound or edge-case).

Closed-world rule: assume nothing outside the customer message.

Return strict JSON:
{
  "auto_handle_safe": <boolean>,
  "reaso

In [9]:
# ============ CELL 7: Generation implementation ============
from support_agent.generation.generator import Generator

GENERATOR_MODEL = "openai/gpt-oss-20b"
GEN_CACHE_PATH  = CACHE_DIR / "generated_replies.jsonl"

def _load_gen_cache(path):
    cache = {}
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    r = json.loads(line)
                    cache[r["key"]] = r["response"]
                except Exception:
                    pass
    return cache

_gen_cache = _load_gen_cache(GEN_CACHE_PATH)
print(f"Generator cache entries: {len(_gen_cache)}")

GENERATOR = Generator(
    model=GENERATOR_MODEL,
    cache_path=GEN_CACHE_PATH,
    cache=_gen_cache,
)
print(f"Generator ready: model={GENERATOR_MODEL}")

# Load corpus and precedent resolver
CORPUS = {}
with open(RET_CORPUS, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        CORPUS[int(r["doc_id"])] = r
print(f"Corpus loaded: {len(CORPUS)} docs")

def get_precedents(top_ids, k: int = 5):
    out = []
    for doc_id in list(top_ids)[:k]:
        d = CORPUS.get(int(doc_id))
        if d is None:
            continue
        out.append({
            "intent":        d.get("intent", "unknown"),
            "customer_text": d.get("customer_text", ""),
            "response_text": d.get("response_text", ""),
        })
    return out

Generator cache entries: 257
Generator ready: model=openai/gpt-oss-20b
Corpus loaded: 50 docs


In [10]:
# ============ CELL 8: Judge implementation + dry-run ============
JUDGE_MODEL      = "openai/gpt-oss-120b"
JUDGE_CACHE_PATH = CACHE_DIR / "judge_scores.jsonl"

def _load_judge_cache(path):
    cache = {}
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    r = json.loads(line)
                    cache[r["key"]] = r["response"]
                except Exception:
                    pass
    return cache

_judge_cache = _load_judge_cache(JUDGE_CACHE_PATH)
print(f"Judge cache entries: {len(_judge_cache)}")

JUDGE = Judge(
    model=JUDGE_MODEL,
    cache_path=JUDGE_CACHE_PATH,
    cache=_judge_cache,
)
print(f"Judge ready: model={JUDGE_MODEL}")

# --- Dry-run: generation ---
print("\nDry-run — generation on 1 dev row (grounded):")
_r = dev.iloc[0]
_prec = get_precedents(_r["top_ids"], k=5)
_gen = GENERATOR.generate(_r["customer_text"], precedents=_prec)
print(f"  api_status: {_gen['api_status']}")
print(f"  reply len:  {len(_gen['reply'])}")
print(f"  grounded:   {_gen['grounded']}")
print(f"  ranks used: {_gen['used_precedent_ranks']}")

# --- Dry-run: judges ---
print("\nDry-run — judges:")
_esc = JUDGE.escalation_suitability(_r["customer_text"])
print(f"  escalation: auto_handle_safe={_esc['auto_handle_safe']} "
      f"reasons={_esc['reasons']}")
if _gen["reply"]:
    _uni = JUDGE.universal(_r["customer_text"], _gen["reply"])
    print(f"  universal:  {_uni}")
    _grd = JUDGE.groundedness(_r["customer_text"], _prec, _gen["reply"])
    print(f"  groundedness: {_grd}")

Judge cache entries: 514
Judge ready: model=openai/gpt-oss-120b

Dry-run — generation on 1 dev row (grounded):
  api_status: ok
  reply len:  115
  grounded:   False
  ranks used: []

Dry-run — judges:
  escalation: auto_handle_safe=False reasons=['Unclear request', 'Insufficient information', 'No actionable question']
  universal:  {'correctness': 5, 'relevance': 5, 'helpfulness': 4, 'unsupported_claim': False, 'unsupported_claims': []}
  groundedness: {'groundedness': 5, 'unsupported_claims': []}


In [11]:
# ============ CELL 9: End-to-end runner ============
def run_one_row(row, *, use_retrieval: bool, apply_escalation: bool,
                force_escalate: bool = False) -> dict:
    """Run one row through the agent. All intermediate state retained."""
    out = {
        "root_id":              int(row["root_id"]),
        "customer_text":        row["customer_text"],
        "true_intent":          row["true_intent"],
        "pred_intent":          row["pred_intent"],
        "confidence":           float(row["confidence"]),
        "confidence_calibrated": float(row["confidence_calibrated"]),
        "alternative_intent":   row.get("alternative_intent", ""),
        "alternative_confidence": float(row.get("alternative_confidence", 0.0)),
        "corpus_coverage":      int(row["corpus_coverage"]),
        "retrieved_doc_ids":    list(row["top_ids"])[:5],
        "retrieval_scores":     list(row["top_scores"])[:5],
        "_source":              row["_source"],
    }

    if force_escalate:
        out.update({
            "escalated": True,
            "escalation_reason": "force_escalate_all",
            "reply": None,
            "used_precedent_ranks": [],
            "grounded": False,
            "api_status": "skipped",
        })
        return out

    if apply_escalation:
        escalated, reason = decide(row, ESCALATION)
    else:
        escalated, reason = False, ""

    if escalated:
        out.update({
            "escalated": True,
            "escalation_reason": reason,
            "reply": None,
            "used_precedent_ranks": [],
            "grounded": False,
            "api_status": "skipped",
        })
        return out

    precedents = get_precedents(row["top_ids"], k=5) if use_retrieval else None
    result = GENERATOR.generate(row["customer_text"], precedents=precedents)

    out.update({
        "escalated": False,
        "escalation_reason": "",
        "reply": result["reply"],
        "used_precedent_ranks": result["used_precedent_ranks"],
        "grounded": result["grounded"],
        "api_status": result["api_status"],
    })
    return out

In [12]:
# ============ CELL 10: Dev run ============
APPROACHES_DEV = {
    "ungrounded":           dict(use_retrieval=False, apply_escalation=False),
    "grounded_no_escalate": dict(use_retrieval=True,  apply_escalation=False),
    "full_pipeline":        dict(use_retrieval=True,  apply_escalation=True),
    "escalate_all":         dict(use_retrieval=False,  apply_escalation=False,
                                  force_escalate=True),
}

# Dry-run on 3 rows
print("Dry-run on 3 dev rows (first approach only):")
for i in range(min(3, len(dev))):
    r = run_one_row(dev.iloc[i], **APPROACHES_DEV["full_pipeline"])
    n = len(r["reply"]) if r["reply"] else 0
    print(f"  row {i}: escalated={r['escalated']} "
          f"reason={r['escalation_reason']!r} reply_len={n}")

# Full dev run
dev_runs = {}
for name, cfg in APPROACHES_DEV.items():
    print(f"\nRunning: {name}")
    rows = []
    t0 = time.time()
    for _, row in dev.iterrows():
        rows.append(run_one_row(row, **cfg))
    elapsed = time.time() - t0
    df = pd.DataFrame(rows)
    dev_runs[name] = df

    out_dir = RUNS_AGENT / name
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_json(out_dir / "dev_replies.jsonl", orient="records",
               lines=True, force_ascii=False)

    n_replies = int(df["reply"].notna().sum())
    n_esc = int(df["escalated"].sum())
    print(f"  rows={len(df)}  replies={n_replies}  escalated={n_esc}  "
          f"t={elapsed:.1f}s")

Dry-run on 3 dev rows (first approach only):
  row 0: escalated=False reason='' reply_len=115
  row 1: escalated=False reason='' reply_len=249
  row 2: escalated=False reason='' reply_len=225

Running: ungrounded
  rows=59  replies=59  escalated=0  t=0.0s

Running: grounded_no_escalate
  rows=59  replies=59  escalated=0  t=0.0s

Running: full_pipeline
  rows=59  replies=51  escalated=8  t=0.0s

Running: escalate_all
  rows=59  replies=0  escalated=59  t=0.0s


In [13]:
# ============ CELL 11: Judge dev ============
# 1) Escalation-suitability on all dev queries (once)
print("Escalation-suitability judge on dev...")
esc_rows = []
for _, row in dev.iterrows():
    res = JUDGE.escalation_suitability(row["customer_text"])
    esc_rows.append({
        "root_id": int(row["root_id"]),
        "auto_handle_safe": res["auto_handle_safe"],
        "reasons": res["reasons"],
    })
esc_df = pd.DataFrame(esc_rows)
esc_df.to_json(RUNS_AGENT / "escalation_suitability_dev.jsonl",
               orient="records", lines=True, force_ascii=False)
print(f"  safe fraction: {esc_df['auto_handle_safe'].mean():.3f}")

# 2) Universal judge on all non-empty replies
print("Universal judge on dev replies...")
uni_rows = []
for name, df in dev_runs.items():
    for _, row in df.iterrows():
        reply = row["reply"]
        if reply is None or (isinstance(reply, float) and pd.isna(reply)):
            continue
        if not str(reply).strip():
            continue
        res = JUDGE.universal(row["customer_text"], reply)
        res["approach"] = name
        res["root_id"] = int(row["root_id"])
        uni_rows.append(res)
uni_df = pd.DataFrame(uni_rows)
uni_df["universal_score"] = uni_df[["correctness", "relevance", "helpfulness"]].sum(axis=1)
uni_df.to_json(RUNS_AGENT / "universal_scores_dev.jsonl",
               orient="records", lines=True, force_ascii=False)
print(f"  judged {len(uni_df)} replies")

# 3) Groundedness judge on grounded replies (B and C only)
print("Groundedness judge on dev grounded replies...")
grd_rows = []
for name in ("grounded_no_escalate", "full_pipeline"):
    df = dev_runs[name]
    for _, row in df.iterrows():
        reply = row["reply"]
        if reply is None or (isinstance(reply, float) and pd.isna(reply)):
            continue
        if not str(reply).strip():
            continue
        prec = get_precedents(row["retrieved_doc_ids"], k=5)
        res = JUDGE.groundedness(row["customer_text"], prec, reply)
        res["approach"] = name
        res["root_id"] = int(row["root_id"])
        grd_rows.append(res)
grd_df = pd.DataFrame(grd_rows)
grd_df.to_json(RUNS_AGENT / "groundedness_scores_dev.jsonl",
               orient="records", lines=True, force_ascii=False)
print(f"  judged {len(grd_df)} grounded replies")

Escalation-suitability judge on dev...
  safe fraction: 0.322
Universal judge on dev replies...
  judged 169 replies
Groundedness judge on dev grounded replies...
  judged 110 grounded replies


In [14]:
# ============ CELL 12: Dev comparison ============
def attach_universal(df, uni_df, name):
    d = df.copy()
    sub = uni_df[uni_df["approach"] == name][
        ["root_id", "universal_score", "unsupported_claim"]
    ]
    return d.merge(sub, on="root_id", how="left")

for name in dev_runs:
    dev_runs[name] = attach_universal(dev_runs[name], uni_df, name)

esc_safe_map = esc_df.set_index("root_id")["auto_handle_safe"]

rows = []
for name, df in dev_runs.items():
    n = len(df)
    replies = df[df["reply"].notna() & (df["reply"].astype(str).str.strip() != "")]
    n_replies = len(replies)
    coverage = n_replies / n if n else 0.0
    esc_rate = df["escalated"].mean()
    mean_q = float(replies["universal_score"].mean()) if n_replies else None
    good_rate = float((replies["universal_score"] >= 12).mean()) if n_replies else None
    bad_rate = float((replies["universal_score"] <= 6).mean()) if n_replies else None
    unsup_rate = float(replies["unsupported_claim"].mean()) if n_replies else None

    df2 = df.copy()
    df2["auto_safe"] = df2["root_id"].map(esc_safe_map)
    auto = df2[df2["escalated"] == False]
    unsafe_auto = auto[auto["auto_safe"] == False]
    unsafe_rate = len(unsafe_auto) / len(auto) if len(auto) else 0.0

    should_esc = df2["auto_safe"] == False
    did_esc = df2["escalated"] == True
    tp = int((should_esc & did_esc).sum())
    fp = int((~should_esc & did_esc).sum())
    fn = int((should_esc & ~did_esc).sum())
    esc_prec = tp / (tp + fp) if (tp + fp) else None
    esc_rec = tp / (tp + fn) if (tp + fn) else None

    rows.append({
        "approach": name,
        "n": n,
        "replies": n_replies,
        "automation_coverage": coverage,
        "escalation_rate": esc_rate,
        "mean_auto_reply_quality": mean_q,
        "good_rate_ge12": good_rate,
        "bad_rate_le6": bad_rate,
        "unsupported_rate": unsup_rate,
        "unsafe_automation_rate": unsafe_rate,
        "unsafe_auto_n": len(unsafe_auto),
        "auto_n": len(auto),
        "escalation_precision": esc_prec,
        "escalation_recall": esc_rec,
    })

comparison = pd.DataFrame(rows)
display(comparison.round(3))
comparison.to_csv(RUNS_AGENT / "comparison_dev.csv",
                  index=False, quoting=1, encoding="utf-8")

# Grounding deltas (A vs B on shared rows)
shared = dev_runs["ungrounded"].merge(
    dev_runs["grounded_no_escalate"],
    on="root_id", suffixes=("_A", "_B")
)[["root_id", "universal_score_A", "universal_score_B",
   "unsupported_claim_A", "unsupported_claim_B"]].dropna()

delta_grounding = float(
    (shared["universal_score_B"] - shared["universal_score_A"]).mean()
)
delta_unsupported = float(
    shared["unsupported_claim_A"].mean() - shared["unsupported_claim_B"].mean()
)
print(f"\nΔ_grounding (B − A):     {delta_grounding:+.3f}")
print(f"Δ_unsupported (A − B):   {delta_unsupported:+.3f}")

# Export stratified human spot check
sample_rows = []
for name in ("ungrounded", "grounded_no_escalate", "full_pipeline"):
    df = dev_runs[name]
    candidates = df[df["reply"].notna() & (df["reply"].astype(str).str.strip() != "")]
    picked = candidates.sample(n=min(5, len(candidates)),
                               random_state=RANDOM_STATE)
    for _, r in picked.iterrows():
        sample_rows.append({
            "approach": name,
            "root_id": int(r["root_id"]),
            "query_text": r["customer_text"],
            "reply_text": r["reply"],
            "judge_universal_score": r.get("universal_score"),
            "human_universal_score": "",
            "judge_auto_handle_safe": bool(esc_safe_map.get(r["root_id"], False)),
            "human_auto_handle_safe": "",
            "notes": "",
        })

# 5 difficult rows (escalated by full_pipeline)
fp = dev_runs["full_pipeline"]
esc_fp = fp[fp["escalated"] == True]
if len(esc_fp):
    picked_esc = esc_fp.sample(n=min(5, len(esc_fp)), random_state=RANDOM_STATE)
    for _, r in picked_esc.iterrows():
        sample_rows.append({
            "approach": "full_pipeline",
            "root_id": int(r["root_id"]),
            "query_text": r["customer_text"],
            "reply_text": "",
            "judge_universal_score": None,
            "human_universal_score": "",
            "judge_auto_handle_safe": bool(esc_safe_map.get(r["root_id"], False)),
            "human_auto_handle_safe": "",
            "notes": "escalated",
        })

spot = pd.DataFrame(sample_rows)
spot.to_csv(RUNS_AGENT / "human_spot_check.csv",
            index=False, quoting=1, encoding="utf-8")
print(f"\nWrote human_spot_check.csv ({len(spot)} rows)")
print(f"  → {RUNS_AGENT / 'human_spot_check.csv'}")
print("PAUSE: fill `human_universal_score` (3–15) and "
      "`human_auto_handle_safe` (y/n), then run Cell 13.")

,approach,n,replies,automation_coverage,escalation_rate,mean_auto_reply_quality,good_rate_ge12,bad_rate_le6,unsupported_rate,unsafe_automation_rate,unsafe_auto_n,auto_n,escalation_precision,escalation_recall
0,ungrounded,59,59,1.000,0.000,13.186,0.864,0.017,0.254,0.678,40,59,NaN,0.0
1,grounded_no_escalate,59,59,1.000,0.000,12.678,0.746,0.017,0.542,0.678,40,59,NaN,0.0
2,full_pipeline,59,51,0.864,0.136,12.824,0.765,0.020,0.510,0.706,36,51,0.500,0.1
3,escalate_all,59,0,0.000,1.000,NaN,NaN,NaN,NaN,0.000,0,0,0.678,1.0



Δ_grounding (B − A):     -0.508
Δ_unsupported (A − B):   -0.288

Wrote human_spot_check.csv (20 rows)
  → D:\CODIN PLAYGROUND\ML-AI\ResolveIQ\runs\agent\human_spot_check.csv
PAUSE: fill `human_universal_score` (3–15) and `human_auto_handle_safe` (y/n), then run Cell 13.


In [19]:
# ============ CELL 13: Human agreement + lock ============
spot = pd.read_csv(RUNS_AGENT / "human_spot_check.csv", encoding="utf-8")

def _nonblank(s):
    return s.notna() & (s.astype(str).str.strip() != "")

graded  = spot[_nonblank(spot["human_universal_score"])].copy()
auto_l  = spot[_nonblank(spot["human_auto_handle_safe"])].copy()

# --- HARD REQUIREMENTS before locking ---
assert len(spot) == 20, f"Expected 20 spot-check rows, got {len(spot)}"
assert auto_l["human_auto_handle_safe"].astype(str).str.strip().ne("").sum() == 20, \
    "All 20 human_auto_handle_safe values must be filled"
assert len(graded) >= 15, \
    f"Need at least 15 human_universal_score values, got {len(graded)}"
print(f"Human spot-check complete: {len(spot)} rows, "
      f"{len(graded)} scored, "
      f"{(auto_l['human_auto_handle_safe'].astype(str).str.strip() != '').sum()} safe-flagged")


if len(graded) >= 3:
    try:
        from scipy.stats import spearmanr
        j = graded["judge_universal_score"].astype(float)
        h = graded["human_universal_score"].astype(float)
        rho, p = spearmanr(j, h)
        mad = float((j - h).abs().mean())
        print(f"Universal judge vs human: n={len(graded)}  "
              f"ρ={rho:.3f} (p={p:.3f})  MAD={mad:.2f}")
    except ImportError:
        print("scipy not installed; skipping Spearman")

if len(auto_l) >= 3:
    h = auto_l["human_auto_handle_safe"].astype(str).str.lower().isin(
        ["y", "yes", "1", "true"]
    )
    j = auto_l["judge_auto_handle_safe"].astype(bool)
    print(f"Escalation-suitability judge vs human: n={len(auto_l)}  "
          f"agreement={(h == j).mean():.3f}")

# --- Lock ---
CHOSEN_AGENT = RUNS_AGENT / "chosen_agent.json"
SAFETY_GATE = 0.10
SIMPLICITY_ORDER = ["ungrounded", "grounded_no_escalate",
                    "full_pipeline", "escalate_all"]

candidates = comparison[~comparison["approach"].isin(["escalate_all"])].copy()
passing = candidates[candidates["unsafe_automation_rate"] <= SAFETY_GATE]
if len(passing) == 0:
    print(f"\nWARNING: no approach passes safety gate {SAFETY_GATE}. "
          f"Proceeding with all candidates.")
    passing = candidates

passing = passing.sort_values("mean_auto_reply_quality", ascending=False)
top_score = passing.iloc[0]["mean_auto_reply_quality"]
within = passing[passing["mean_auto_reply_quality"] >= top_score - 0.30]

chosen_name = None
for s in SIMPLICITY_ORDER:
    if s in within["approach"].values:
        chosen_name = s
        break
if chosen_name is None:
    chosen_name = passing.iloc[0]["approach"]

print(f"\nChosen approach: {chosen_name}")

chosen_payload = {
    "approach": chosen_name,
    "escalation_config": ESCALATION.to_dict(),
    "generator_model": GENERATOR_MODEL,
    "judge_model": JUDGE_MODEL,
    "safety_gate": SAFETY_GATE,
    "retrieval_top_k": 5,
    "selection_rule": {
        "primary": "mean_universal_non_escalated",
        "tie_delta": 0.30,
        "tie_break": "higher_coverage_then_simplicity",
        "simplicity_order": SIMPLICITY_ORDER,
    },
    "escalation_rubric_sha256":   ESCALATION_RUBRIC_SHA,
    "universal_rubric_sha256":    UNIVERSAL_RUBRIC_SHA,
    "groundedness_rubric_sha256": GROUNDEDNESS_RUBRIC_SHA,
    "taxonomy_sha256":    tax_sha,
    "golden_sha256":      golden_sha,
    "classifier_sha256":  cls_chosen_sha,
    "retriever_sha256":   ret_chosen_sha,
    "corpus_sha256":      CORPUS_SHA,
    "delta_grounding":    delta_grounding,
    "delta_unsupported":  delta_unsupported,
    "locked_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "note": "Immutable lock written before test evaluation.",
}
CHOSEN_AGENT.write_text(json.dumps(chosen_payload, indent=2), encoding="utf-8")
chosen_agent_sha = hashlib.sha256(CHOSEN_AGENT.read_bytes()).hexdigest()
print(f"chosen_agent.json SHA: {chosen_agent_sha[:16]}...")

Human spot-check complete: 20 rows, 15 scored, 20 safe-flagged
Universal judge vs human: n=15  ρ=0.722 (p=0.002)  MAD=0.67
Escalation-suitability judge vs human: n=20  agreement=0.950


Chosen approach: ungrounded
chosen_agent.json SHA: dc6c09f6f3cd147e...


In [20]:
# ============ CELL 14: Load TEST ============
test_rows = []
with open(GOLDEN_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r["split"] == "test":
            test_rows.append(r)
test_golden = pd.DataFrame(test_rows)[["root_id", "customer_text",
                                       "true_intent", "_source"]].copy()
test_golden["root_id"] = test_golden["root_id"].astype(int)

cls_test = pd.read_json(CLS_TEST_PREDS, orient="records", lines=True)
cls_test = cls_test[["root_id", "pred_intent", "confidence",
                     "alternative_intent", "alternative_confidence"]].copy()
cls_test["root_id"] = cls_test["root_id"].astype(int)

ret_test = pd.read_json(RET_TEST_PREDS, orient="records", lines=True)
ret_test = ret_test[["root_id", "corpus_coverage", "top_ids", "top_scores"]].copy()
ret_test["root_id"] = ret_test["root_id"].astype(int)

test = (test_golden
        .merge(cls_test, on="root_id", how="inner")
        .merge(ret_test, on="root_id", how="inner"))
assert len(test) == GOLDEN_TEST_SIZE

test["confidence_calibrated"] = test["confidence"].apply(
    lambda c: calibrate_confidence(c, REL_TABLE)
)

print(f"Test rows: {len(test)}")
print(test["true_intent"].value_counts().to_string())

Test rows: 139
true_intent
on_board_issue        23
delay_compensation    18
service_disruption    18
other                 18
booking_issue         14
praise_or_chatter     11
refund_request        10
timetable_info        10
seat_reservation       9
lost_property          7
ambiguous              1


In [24]:
# ============ CELL 15: One-shot test run ============
TEST_LOCK  = RUNS_AGENT / "TEST_LOCK.json"
TEST_PREDS = RUNS_AGENT / "predictions_test.jsonl"

if TEST_LOCK.exists():
    lock = json.loads(TEST_LOCK.read_text(encoding="utf-8"))
    if lock["chosen_agent_sha"] != chosen_agent_sha:
        raise RuntimeError("Different chosen agent — test already evaluated")
    assert hashlib.sha256(TEST_PREDS.read_bytes()).hexdigest() == \
        lock["predictions_sha256"]
    print("Test already frozen. No generation will be re-run.")
    raise SystemExit

print(f"First test evaluation. Chosen: {chosen_name}")
CHOSEN_CFG = APPROACHES_DEV[chosen_name]

t0 = time.time()
test_rows_out = []
for _, row in test.iterrows():
    test_rows_out.append(run_one_row(row, **CHOSEN_CFG))
elapsed = time.time() - t0

test_df_out = pd.DataFrame(test_rows_out)
test_df_out.to_json(TEST_PREDS, orient="records", lines=True,
                    force_ascii=False)

n_replies = int(test_df_out["reply"].notna().sum())
n_esc = int(test_df_out["escalated"].sum())

lock_payload = {
    "chosen_agent_sha": chosen_agent_sha,
    "predictions_sha256": hashlib.sha256(TEST_PREDS.read_bytes()).hexdigest(),
    "evaluated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
TEST_LOCK.write_text(json.dumps(lock_payload, indent=2), encoding="utf-8")

print(f"Test complete: rows={len(test_df_out)} replies={n_replies} "
      f"escalated={n_esc} t={elapsed:.1f}s")
print("TEST_LOCK written. Do not re-run this cell.")

First test evaluation. Chosen: ungrounded
Test complete: rows=139 replies=139 escalated=0 t=0.0s
TEST_LOCK written. Do not re-run this cell.


In [25]:
# ============ CELL 16: Judge test + freeze + manifest ============
# 1) Escalation-suitability on all test queries
print("Escalation-suitability judge on test...")
esc_test = []
for _, row in test.iterrows():
    res = JUDGE.escalation_suitability(row["customer_text"])
    esc_test.append({
        "root_id": int(row["root_id"]),
        "auto_handle_safe": res["auto_handle_safe"],
        "reasons": res["reasons"],
    })
esc_test_df = pd.DataFrame(esc_test)
esc_test_df.to_json(RUNS_AGENT / "escalation_suitability_test.jsonl",
                    orient="records", lines=True, force_ascii=False)

# 2) Universal judge on generated replies
print("Universal judge on test replies...")
uni_test = []
for _, row in test_df_out.iterrows():
    reply = row["reply"]
    if reply is None or (isinstance(reply, float) and pd.isna(reply)):
        continue
    if not str(reply).strip():
        continue
    res = JUDGE.universal(row["customer_text"], reply)
    res["root_id"] = int(row["root_id"])
    uni_test.append(res)
uni_test_df = pd.DataFrame(uni_test)
uni_test_df["universal_score"] = uni_test_df[
    ["correctness", "relevance", "helpfulness"]
].sum(axis=1)
uni_test_df.to_json(RUNS_AGENT / "universal_scores_test.jsonl",
                    orient="records", lines=True, force_ascii=False)

# 3) Groundedness on grounded replies
grd_test = []
if chosen_name in ("grounded_no_escalate", "full_pipeline"):
    print("Groundedness judge on test grounded replies...")
    for _, row in test_df_out.iterrows():
        reply = row["reply"]
        if reply is None or (isinstance(reply, float) and pd.isna(reply)):
            continue
        if not str(reply).strip():
            continue
        prec = get_precedents(row["retrieved_doc_ids"], k=5)
        res = JUDGE.groundedness(row["customer_text"], prec, reply)
        res["root_id"] = int(row["root_id"])
        grd_test.append(res)
grd_test_df = pd.DataFrame(grd_test)
if len(grd_test_df):
    grd_test_df.to_json(RUNS_AGENT / "groundedness_scores_test.jsonl",
                        orient="records", lines=True, force_ascii=False)

# --- Final metrics ---
test_merged = test_df_out.merge(
    uni_test_df[["root_id", "universal_score", "unsupported_claim"]],
    on="root_id", how="left",
)
esc_safe_test = esc_test_df.set_index("root_id")["auto_handle_safe"]
test_merged["auto_safe"] = test_merged["root_id"].map(esc_safe_test)

auto = test_merged[test_merged["escalated"] == False]
auto_scored = auto[auto["universal_score"].notna()]

final = {
    "chosen_approach":        chosen_name,
    "n_test":                 int(len(test_merged)),
    "n_replies":              int(auto["reply"].notna().sum()),
    "n_escalated":            int(test_merged["escalated"].sum()),
    "automation_coverage":    float(len(auto) / len(test_merged)),
    "mean_universal_auto":    float(auto_scored["universal_score"].mean())
                              if len(auto_scored) else None,
    "unsupported_rate_auto":  float(auto_scored["unsupported_claim"].mean())
                              if len(auto_scored) else None,
    "unsafe_automation_rate": float((auto["auto_safe"] == False).mean())
                              if len(auto) else 0.0,
    "unsafe_auto_n":          int((auto["auto_safe"] == False).sum()),
    "auto_n":                 int(len(auto)),
    "escalation_rate":        float(test_merged["escalated"].mean()),
}
if len(grd_test_df):
    final["mean_groundedness"] = float(grd_test_df["groundedness"].mean())

# --- Q3: escalation precision/recall on test ---
should_esc = test_merged["auto_safe"] == False
did_esc    = test_merged["escalated"] == True
tp = int((should_esc & did_esc).sum())
fp = int((~should_esc & did_esc).sum())
fn = int((should_esc & ~did_esc).sum())
final["escalation_precision"] = tp / (tp + fp) if (tp + fp) else None
final["escalation_recall"]    = tp / (tp + fn) if (tp + fn) else None
final["escalation_tp"]        = tp
final["escalation_fp"]        = fp
final["escalation_fn"]        = fn

FINAL_METRICS = RUNS_AGENT / "final_metrics.json"
FINAL_METRICS.write_text(json.dumps(final, indent=2), encoding="utf-8")

# --- Manifest ---
MANIFEST = RUNS_AGENT / "artifact_manifest.json"
files = []
for path in sorted(RUNS_AGENT.rglob("*")):
    if path.is_file() and path.name != "artifact_manifest.json":
        files.append({
            "path":       str(path.relative_to(PROJECT_ROOT)),
            "size_bytes": path.stat().st_size,
            "sha256":     hashlib.sha256(path.read_bytes()).hexdigest(),
        })
MANIFEST.write_text(json.dumps({
    "created_at":       datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "chosen_agent_sha": chosen_agent_sha,
    "artifacts":        files,
}, indent=2), encoding="utf-8")

# --- Handoff to 07 ---
handoff = {
    "from_notebook": "06_agent.ipynb",
    "to_notebook":   "07_evaluation.ipynb",
    "consumes": [
        "runs/agent/predictions_test.jsonl",
        "runs/agent/universal_scores_test.jsonl",
        "runs/agent/escalation_suitability_test.jsonl",
        "runs/agent/final_metrics.json",
        "runs/agent/chosen_agent.json",
    ],
    "does_not_rerun_generation": True,
    "does_not_rerun_judging":    True,
    "verification_required_by_07": {
        "taxonomy_sha256":   tax_sha,
        "golden_sha256":     golden_sha,
        "classifier_sha256": cls_chosen_sha,
        "retriever_sha256":  ret_chosen_sha,
        "corpus_sha256":     CORPUS_SHA,
        "chosen_agent_sha":  chosen_agent_sha,
    },
    "notes": (
        "Retrieval scores are ordinal. Reply quality measured by three "
        "independent judges (universal, groundedness, escalation-suitability). "
        "Escalation-suitability label is independent of generation and sees "
        "only the customer message."
    ),
}
(RUNS_AGENT / "handoff_to_07.json").write_text(
    json.dumps(handoff, indent=2), encoding="utf-8"
)

print(f"\n=== TEST METRICS ({chosen_name}) ===")
for k, v in final.items():
    if isinstance(v, float):
        print(f"  {k:28s} {v:.4f}")
    else:
        print(f"  {k:28s} {v}")
print(f"\nArtifacts: {len(files)} files")

Escalation-suitability judge on test...
Universal judge on test replies...

=== TEST METRICS (ungrounded) ===
  chosen_approach              ungrounded
  n_test                       139
  n_replies                    139
  n_escalated                  0
  automation_coverage          1.0000
  mean_universal_auto          13.3741
  unsupported_rate_auto        0.1942
  unsafe_automation_rate       0.5971
  unsafe_auto_n                83
  auto_n                       139
  escalation_rate              0.0000
  escalation_precision         None
  escalation_recall            0.0000
  escalation_tp                0
  escalation_fp                0
  escalation_fn                83

Artifacts: 17 files


In [26]:
# ============ CELL 16b: Evaluation lock (post-judging) ============
# Locks the test-time judging artifacts so that 07_evaluation reads a
# fully-immutable package. Prevents silent re-judging on rerun.

EVAL_LOCK = RUNS_AGENT / "EVAL_LOCK.json"

def _sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

test_artifacts = {
    "chosen_agent":           RUNS_AGENT / "chosen_agent.json",
    "predictions_test":       RUNS_AGENT / "predictions_test.jsonl",
    "test_lock":              RUNS_AGENT / "TEST_LOCK.json",
    "escalation_suitability_test": RUNS_AGENT / "escalation_suitability_test.jsonl",
    "universal_scores_test":  RUNS_AGENT / "universal_scores_test.jsonl",
    "final_metrics":          RUNS_AGENT / "final_metrics.json",
}
if (RUNS_AGENT / "groundedness_scores_test.jsonl").exists():
    test_artifacts["groundedness_scores_test"] = \
        RUNS_AGENT / "groundedness_scores_test.jsonl"

payload = {
    "created_at":  datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "artifacts":   {name: _sha(p) for name, p in test_artifacts.items()},
    "rubric_sha256": {
        "escalation":   ESCALATION_RUBRIC_SHA,
        "universal":    UNIVERSAL_RUBRIC_SHA,
        "groundedness": GROUNDEDNESS_RUBRIC_SHA,
    },
    "judge_model":  JUDGE_MODEL,
}

if EVAL_LOCK.exists():
    existing = json.loads(EVAL_LOCK.read_text(encoding="utf-8"))
    if existing["artifacts"] != payload["artifacts"]:
        raise RuntimeError(
            "Evaluation artifacts changed since lock was written. "
            "Do not modify test-time artifacts after EVAL_LOCK.json exists."
        )
    print("EVAL_LOCK.json already present and matches current artifacts.")
else:
    EVAL_LOCK.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Wrote EVAL_LOCK.json with {len(payload['artifacts'])} artifact hashes")
    for name, sha in payload["artifacts"].items():
        print(f"  {name:32s} {sha[:16]}...")

Wrote EVAL_LOCK.json with 6 artifact hashes
  chosen_agent                     dc6c09f6f3cd147e...
  predictions_test                 ab08bce7efb7d04d...
  test_lock                        ccfc41117caafda4...
  escalation_suitability_test      7b54414e48e0136a...
  universal_scores_test            504cdb2c4d0f04f8...
  final_metrics                    83d2efe7a84e6775...


In [27]:
# ============ CELL 17: Failure analysis ============
test_preds  = pd.read_json(TEST_PREDS, orient="records", lines=True)
test_judged = test_preds.merge(
    uni_test_df[["root_id", "universal_score", "unsupported_claim"]],
    on="root_id", how="left",
)
esc_map_test = esc_test_df.set_index("root_id")["auto_handle_safe"]
test_judged["auto_safe"] = test_judged["root_id"].map(esc_map_test)
test_judged["classifier_correct"] = (
    test_judged["true_intent"] == test_judged["pred_intent"]
)

auto = test_judged[test_judged["escalated"] == False].copy()
bad = auto[
    auto["universal_score"].notna() & (auto["universal_score"] <= 6)
].copy()

def attribute(row):
    if not bool(row["classifier_correct"]):
        return "classification_error"
    if int(row["corpus_coverage"]) == 0:
        return "retrieval_coverage_missing"
    if row["universal_score"] is not None and row["universal_score"] <= 6:
        if pd.notna(row.get("unsupported_claim")) and row["unsupported_claim"]:
            return "generation_unsupported_claim"
        return "retrieval_present_but_reply_bad"
    return "ok"

if len(bad):
    bad["attribution"] = bad.apply(attribute, axis=1)
    bad.to_csv(RUNS_AGENT / "failure_analysis_test.csv",
               index=False, quoting=1, encoding="utf-8")
    print(f"Bad auto-handled rows: {len(bad)} / {len(auto)}")
    print()
    print("Attribution:")
    print(bad["attribution"].value_counts().to_string())
else:
    print("No bad auto-handled rows (score ≤ 6).")

print()
esc = test_judged[test_judged["escalated"] == True]
if len(esc):
    print("Escalation reasons on test:")
    print(esc["escalation_reason"].value_counts().to_string())

print()
print(f"Classifier accuracy on test: "
      f"{test_judged['classifier_correct'].mean():.3f}")
print(f"Automation coverage on test: "
      f"{(test_judged['escalated'] == False).mean():.3f}")

No bad auto-handled rows (score ≤ 6).


Classifier accuracy on test: 0.604
Automation coverage on test: 1.000


In [28]:
# ============ DIAGNOSTIC: What separates safe from unsafe? ============
# Loads existing dev artifacts. Does not re-run anything.

import pandas as pd
import numpy as np
from pathlib import Path

RA = Path.cwd().parent / "runs" / "agent"

# Reload dev pieces
esc_dev = pd.read_json(RA / "escalation_suitability_dev.jsonl",
                       orient="records", lines=True)
uni_dev = pd.read_json(RA / "universal_scores_dev.jsonl",
                       orient="records", lines=True)

# Join with dev (which has classifier + retriever predictions)
diag = dev.merge(esc_dev[["root_id", "auto_handle_safe"]], on="root_id", how="left")

# Also merge in the full_pipeline escalated decision from dev_runs
fp = dev_runs["full_pipeline"][["root_id", "escalated", "escalation_reason"]]
diag = diag.merge(fp, on="root_id", how="left")

# Compute margin
diag["margin"] = diag["confidence"] - diag["alternative_confidence"]

print("=" * 72)
print("SAFE vs UNSAFE — distribution comparison (dev, n=59)")
print("=" * 72)

for label in [True, False]:
    sub = diag[diag["auto_handle_safe"] == label]
    tag = "SAFE" if label else "UNSAFE"
    print(f"\n{tag} (n={len(sub)}):")
    print(f"  confidence_raw:        mean={sub['confidence'].mean():.3f}  "
          f"median={sub['confidence'].median():.3f}")
    print(f"  confidence_calibrated: mean={sub['confidence_calibrated'].mean():.3f}  "
          f"median={sub['confidence_calibrated'].median():.3f}")
    print(f"  margin:                mean={sub['margin'].mean():.3f}  "
          f"median={sub['margin'].median():.3f}")
    print(f"  corpus_coverage=1:     {(sub['corpus_coverage']==1).sum()} / {len(sub)}")
    print(f"  pred_intent in other/ambiguous: "
          f"{sub['pred_intent'].isin(['other','ambiguous']).sum()} / {len(sub)}")

print("\n" + "=" * 72)
print("Escalation rule coverage by outcome (full_pipeline)")
print("=" * 72)
print(diag.groupby(["auto_handle_safe", "escalated"]).size()
      .rename("n").to_frame().to_string())

print("\n" + "=" * 72)
print("Escalation reasons that fired")
print("=" * 72)
print(diag[diag["escalated"] == True]["escalation_reason"].value_counts().to_string())

# Save for the report
diag.to_csv(RA / "diagnostic_safe_vs_unsafe.csv",
            index=False, quoting=1, encoding="utf-8")
print(f"\nWrote diagnostic_safe_vs_unsafe.csv ({len(diag)} rows)")

SAFE vs UNSAFE — distribution comparison (dev, n=59)

SAFE (n=19):
  confidence_raw:        mean=0.939  median=0.950
  confidence_calibrated: mean=0.602  median=0.602
  margin:                mean=0.883  median=0.900
  corpus_coverage=1:     18 / 19
  pred_intent in other/ambiguous: 3 / 19

UNSAFE (n=40):
  confidence_raw:        mean=0.918  median=0.920
  confidence_calibrated: mean=0.626  median=0.602
  margin:                mean=0.827  median=0.840
  corpus_coverage=1:     40 / 40
  pred_intent in other/ambiguous: 4 / 40

Escalation rule coverage by outcome (full_pipeline)
                             n
auto_handle_safe escalated    
False            False      36
                 True        4
True             False      15
                 True        4

Escalation reasons that fired
escalation_reason
intent_other_or_ambiguous    7
corpus_coverage_zero         1

Wrote diagnostic_safe_vs_unsafe.csv (59 rows)


In [29]:
# ============ DIAGNOSTIC: Grounding failures ============
import pandas as pd
from pathlib import Path

RA = Path.cwd().parent / "runs" / "agent"

u = dev_runs["ungrounded"][["root_id", "customer_text", "reply",
                             "universal_score"]].rename(
    columns={"reply": "reply_ungrounded",
             "universal_score": "score_ungrounded"})
g = dev_runs["grounded_no_escalate"][["root_id", "reply",
                                       "universal_score",
                                       "retrieved_doc_ids"]].rename(
    columns={"reply": "reply_grounded",
             "universal_score": "score_grounded"})
cmp = u.merge(g, on="root_id")
cmp["delta"] = cmp["score_grounded"] - cmp["score_ungrounded"]

worse = cmp.nsmallest(10, "delta")

for _, r in worse.iterrows():
    print("=" * 72)
    print(f"root_id:      {r['root_id']}")
    print(f"delta:        {r['delta']:+.2f}")
    print(f"query:        {r['customer_text'][:200]}")
    print(f"UNGROUNDED ({r['score_ungrounded']:.0f}):")
    print(f"  {r['reply_ungrounded']}")
    print(f"GROUNDED ({r['score_grounded']:.0f}):")
    print(f"  {r['reply_grounded']}")
    print(f"precedents used: {r['retrieved_doc_ids'][:3]}")
    print()

cmp.to_csv(RA / "diagnostic_grounding_failures.csv",
           index=False, quoting=1, encoding="utf-8")
print(f"Saved {len(cmp)} rows to diagnostic_grounding_failures.csv")

root_id:      59200
delta:        -7.00
query:        #really https://t.co/ICrWwBRkY6
UNGROUNDED (13):
  Thank you for reaching out. I’m not sure what you’re referring to with the link. Could you please provide more details or let me know how I can help? For further assistance, please contact our customer support team via the official GWRHelp channels.
GROUNDED (6):
  I’m sorry you’re experiencing this disruption. The service was cancelled due to congestion, which can be frustrating. If you need further assistance, please let us know.
precedents used: [54019]

root_id:      1014147
delta:        -6.00
query:        @44464 1928 Didcot - Bristol parkway I can’t sit in my reserved seat, can I get a refund? Heard something on tannoy but didn’t understand!
UNGROUNDED (15):
  Thank you for contacting us. I’m sorry you’re having trouble with your reserved seat. I’m not able to process refunds directly. Please contact our Customer Care team via the GWR app or call 0344 500 5000 for assistance 